# 渲染与性能

学习目标：能根据渲染阶段定位成本，用相同负载比较改动，并识别渲染隔离和按需渲染的行为边界。

前置知识：盒模型、定位、变换、关键帧、媒体查询、外部资源与开发者工具。

适用范围：浏览器性能工具以 Chrome 为例；contain 与 content-visibility 按所用值核对目标版本。仅资源页用少量 JavaScript 触发延后显示，不依赖外部网络和第三方字体。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/21-rendering-and-performance/。

1. [index.html](scripts/21-rendering-and-performance/index.html)、[performance.css](scripts/21-rendering-and-performance/performance.css)：相同位移对照、透明度和可选提示。
2. [resources.html](scripts/21-rendering-and-performance/resources.html)、[resources.css](scripts/21-rendering-and-performance/resources.css)：本地字体观察与空间预留对照。
3. [delayed-content.js](scripts/21-rendering-and-performance/delayed-content.js)、[diagram.svg](scripts/21-rendering-and-performance/diagram.svg)：延后显示的辅助触发与本例绘制的本地图片。
4. [containment.html](scripts/21-rendering-and-performance/containment.html)、[containment.css](scripts/21-rendering-and-performance/containment.css)：绘制隔离与尺寸隔离边界。
5. [long.html](scripts/21-rendering-and-performance/long.html)、[baseline.html](scripts/21-rendering-and-performance/baseline.html)、[long.css](scripts/21-rendering-and-performance/long.css)：相同长内容的按需渲染和普通渲染对照。

## 打开配套页面

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/21-rendering-and-performance/index.html)。

服务根目录是 content/Web与应用开发/css；修改文件后保存并刷新

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 从样式变化到屏幕更新

样式计算确定匹配规则与最终属性；布局（layout）确定盒子的尺寸和位置；绘制（paint）形成文字、边框、背景等绘制工作，栅格化（rasterization）把绘制内容转为像素；合成（compositing）把已绘制的内容按位置和层次组合成画面。

不是每一帧都完整执行全部阶段。尺寸或几何位置变化可能要求布局和后续处理；颜色变化通常需要重新绘制；在合适条件下，transform 或 opacity 的更新可能复用已绘制内容完成合成。缓存、内容、引擎和设备会影响具体路径。

本例用同样的16个矩形、2s时长和160px位移比较 left 与 transform。left 用于相对定位，保留正常流占位，但几何位置更新仍可能产生布局工作；transform 改绘制坐标。二者视觉目标相同，才适合用来比较实现成本。

切换模式前先暂停；检查终点和往返路径一致，再开始测量。不能把“没有肉眼卡顿”当作“没有布局或绘制”。

```html
<input id="left-mode" name="mode" type="radio" checked><label for="left-mode">left 位移</label>
<input id="transform-mode" name="mode" type="radio"><label for="transform-mode">transform 位移</label>
<input id="run" type="checkbox"><label for="run">播放</label>
<input id="hint" type="checkbox"><label for="hint">仅第一项启用 will-change</label>
```

```css
@keyframes move-left { from { left: 0; } to { left: 160px; } }
@keyframes move-transform { from { transform: translateX(0); } to { transform: translateX(160px); } }
.samples .row { height: 24px; }
.mover {
  position: relative;
  width: 40px;
  height: 16px;
  background: #286276;
  animation: move-left 2s linear infinite alternate paused;
}
#transform-mode:checked ~ .samples .mover { animation-name: move-transform; }
#run:checked ~ .samples .mover { animation-play-state: running; }
/* 两种模式都在2s内移动160px；每次只播放一种，默认暂停。 */
```

配套文件：[index.html](scripts/21-rendering-and-performance/index.html)、[performance.css](scripts/21-rendering-and-performance/performance.css) · [浏览器预览](http://127.0.0.1:8101/scripts/21-rendering-and-performance/index.html)

## 2 本章使用的属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| transform | 变换 | 位移实现对照 |
| left | 左侧偏移 | 几何位置更新对照 |
| opacity | 不透明度 | 整体透明度变化 |
| will-change | 预期变化提示 | 按需尝试浏览器优化 |
| contain | 渲染隔离 | 限定尺寸、布局、绘制等关系 |
| content-visibility | 内容可见性处理 | 允许跳过不相关内容渲染 |
| contain-intrinsic-size | 隔离时的替代固有尺寸 | 估计或记住跳过内容的尺寸 |
| aspect-ratio | 首选宽高比 | 为延后内容预留空间 |
| font-family | 字体族 | 选择字体与回退 |
| line-height | 行高 | 稳定文字行框 |

## 3 样式表和字体也是性能输入

适用于当前页面的普通外部样式表通常会阻塞首次渲染，浏览器需要知道样式才能正确显示。先在 Network 看 CSS 的发现时间、请求链、传输体积与缓存，避免把所有首次显示慢都归咎于选择器。

减少未使用样式、避免很深的 @import 发现链，可能减少加载工作；将关键 CSS 内联又会增加 HTML 体积并影响复用缓存。这些是候选改动，应该按实际页面测量取舍，不是要求每个教学页建立构建流水线。

字体通常要等样式确实使用某个 @font-face 后才按需要加载。字体到达过晚可能延迟文字显示，回退字体与目标字体度量不同还可能改变换行。font-display 是 @font-face 的描述符：swap 优先显示回退后再交换，optional 更限制后期替换；它们都不保证零布局偏移，也不替代字体回退设计。

本页只用本地字体候选，避免引入第三方字体资产；在 Computed 的 Rendered Fonts 区域查看实际使用字体，在 Network 确认没有外部字体请求。可临时换为 sans-serif 观察文字度量，但这不是网络字体加载性能实测。

测真实网站时再使用已有合法字体资产：分别记录禁用缓存的冷加载与允许缓存的再次加载，比较 CSS 与字体 Timing、文字出现时间和换行，不要混在一份结论中。

```html
<p class="font-sample">Font metrics: WWW iii 0123456789；字体度量会影响换行。</p>
```

```css
.font-sample { max-width: 320px; font-family: Georgia, "Times New Roman", serif; line-height: 1.6; }
/* 这些是本地字体候选，不下载字体；实际命中的字体取决于系统与字符覆盖。 */
```

配套文件：[resources.html](scripts/21-rendering-and-performance/resources.html)、[resources.css](scripts/21-rendering-and-performance/resources.css) · [浏览器预览](http://127.0.0.1:8101/scripts/21-rendering-and-performance/resources.html)

## 4 为延后内容预留空间

布局偏移是已有可见内容的位置发生变化。图片、动态插入内容或字体替换都可能成为原因；工具列出的被移动元素不一定是根因，要找是谁在它前面改变了空间。

本例按钮通过 delayed-content.js 至少等待约两秒后给 .examples 添加 is-ready 类，让两张图片同时由 display: none 变为 block。这模拟内容晚到后的显示，不模拟下载速率；setTimeout 不是精确时钟。图片使用本章自绘的本地 SVG。

两张图片都有 HTML 的 width、height 尺寸信息；但在 display: none 期间它们不占空间，仍需可见外层容器预留位置。右组的 aspect-ratio: 3 / 1 与图片比例一致，左组没有预留。

录制时保持窗口不滚动且两组都在视口中，点击一次，观察 Layout Shifts 轨道与被移动文字。紧随用户输入的某些偏移不会计入 CLS；这个延迟例子超过常见的500ms输入宽限期，但是否计分仍看实际记录，不能把像素位移直接当作 CLS。

预留值必须贴近实际内容；固定过小高度再裁切文字不是优化。此处展示几何差异，不提供预填分数。

```html
<button id="reveal" type="button">等待两秒后显示图片</button>
<div class="examples">
  <section class="example unreserved">
    <h2>未预留空间</h2>
    <div class="slot"><img src="diagram.svg" width="600" height="200" alt="三个并排矩形"></div>
    <p class="following">观察这段文字的位置。</p>
  </section>
  <section class="example reserved">
    <h2>预留空间</h2>
    <div class="slot"><img src="diagram.svg" width="600" height="200" alt="三个并排矩形"></div>
    <p class="following">观察这段文字的位置。</p>
  </section>
</div>
<script src="delayed-content.js"></script>
```

```css
.examples { display: flex; flex-wrap: wrap; gap: 24px; }
.example { width: 300px; max-width: 100%; }
.slot { width: 100%; background: #eef2f4; }
.slot img { display: none; width: 100%; height: auto; }
.reserved .slot { aspect-ratio: 3 / 1; }
.examples.is-ready .slot img { display: block; }
/* 同一延迟触发两组显示；预留区维持宽高比，未预留组会把后续文字向下推。 */
```

配套文件：[resources.html](scripts/21-rendering-and-performance/resources.html)、[resources.css](scripts/21-rendering-and-performance/resources.css) · [浏览器预览](http://127.0.0.1:8101/scripts/21-rendering-and-performance/resources.html)

## 5 contain 的收益来自明确边界

contain 向浏览器声明某些内部关系可以独立处理，但也会改变布局和绘制语义。按需求选择值，不能给所有容器直接添加 strict。

- size：计算盒子的尺寸时不依赖内部内容；自动块尺寸可能缩为零，需要显式尺寸、最小尺寸或合适的替代固有尺寸。
- inline-size：只在行内轴做尺寸隔离，与 size 不能同时指定。
- layout：建立独立的布局上下文，改变部分定位与格式化关系；外层可用空间仍参与容器尺寸计算，不能理解为与外界所有布局都无关。
- paint：后代绘制限制在容器的溢出裁剪边界，默认通常对应内边距盒；浮层和阴影可能被裁掉。
- style：限制计数器等会影响外部的样式效果，不阻止选择器匹配或字体颜色继承，也不等同 @scope。

content 相当于 layout paint style；strict 还包含 size。layout 或 paint 等隔离还可能建立包含块与层叠上下文，需复查定位浮层。示例刻意保留尺寸塌缩与裁剪对照，不能直接复制成普通卡片样式。

```html
<div class="paint-box"><div class="overflow-piece">未隔离的溢出</div></div>
<div class="paint-box clipped"><div class="overflow-piece">绘制隔离后的溢出</div></div>
<div class="size-shell"><div class="size-box"><div class="sized-content">子内容高80px</div></div></div>
<div class="size-shell"><div class="size-box explicit"><div class="sized-content">额外预留90px</div></div></div>
```

```css
.paint-box { width: 180px; height: 80px; border: 2px solid #246; margin-block: 24px; }
.overflow-piece { width: 260px; height: 40px; background: #cce8ed; }
.clipped { contain: paint; }
.size-shell { min-height: 110px; }
.size-box { contain: size; width: 180px; border: 2px solid #a35e28; }
.sized-content { height: 80px; }
.explicit { min-height: 90px; }
/* paint裁掉后代外溢；size使自动块尺寸不由后代撑开，最后一组显式预留空间。 */
```

配套文件：[containment.html](scripts/21-rendering-and-performance/containment.html)、[containment.css](scripts/21-rendering-and-performance/containment.css) · [浏览器预览](http://127.0.0.1:8101/scripts/21-rendering-and-performance/containment.html)

## 6 content-visibility 与长页面

content-visibility: auto 允许浏览器在内容对用户暂不相关时跳过其内部部分样式、布局和绘制工作。何时提前开始渲染由浏览器决定，不保证元素刚出视口就立即跳过。

它不删除 DOM，不是列表虚拟化，也不保证跳过脚本执行、事件监听或所有资源下载。auto 的内容仍应能被页内查找、焦点与选择操作访问，并在需要时呈现；hidden 则跳过内容且不向这些交互提供它们，不能当作 auto 的同义值。display: none 也不删除 DOM，它主要改变盒生成。

contain-intrinsic-size: auto 480px 提供跳过时的初始尺寸估计，并在正常显示后使用记住的尺寸。这里 480px 是人为选定的布局估计，不是性能数字；估计与实际高度不符时，滚动条范围和位置仍可能调整。

两份长页包含相同的32个条目，增强页外层为 .optimized；只有该类触发下面的规则。基线页保持普通渲染，不支持组合条件时增强页也正常展开。不要通过给内容加 aria-hidden 来实现性能优化。

反复读取跳过子树的尺寸可能迫使浏览器补做渲染工作；测量时避免边滚动边用脚本轮询所有后代。对于包含有意隐藏内容的真实组件，还需单独检查可访问性树与焦点，不能仅依据 auto 就承诺所有组合无差异。

```html
<article class="entry" id="entry-01"><h2>长页条目 01</h2><p>第01节观察段落1：浏览器滚动到这里时，内容仍应正常排版并可以阅读。此处的文字与基线页面完全相同。</p>
<p>第01节观察段落2：浏览器滚动到这里时，内容仍应正常排版并可以阅读。此处的文字与基线页面完全相同。</p>
<p>第01节观察段落3：浏览器滚动到这里时，内容仍应正常排版并可以阅读。此处的文字与基线页面完全相同。</p>
<p>第01节观察段落4：浏览器滚动到这里时，内容仍应正常排版并可以阅读。此处的文字与基线页面完全相同。</p>
<p>第01节观察段落5：浏览器滚动到这里时，内容仍应正常排版并可以阅读。此处的文字与基线页面完全相同。</p>
<p>第01节观察段落6：浏览器滚动到这里时，内容仍应正常排版并可以阅读。此处的文字与基线页面完全相同。</p><a href="#page-top">返回页面顶部</a></article>
```

```css
.entry { border: 1px solid #778899; padding: 16px; margin-block: 16px; }
@supports (content-visibility: auto) and (contain-intrinsic-size: auto 480px) {
  .optimized .entry {
    content-visibility: auto;
    contain-intrinsic-size: auto 480px;
  }
}
/* 480px是本例初始估计，不是实测耗时；显示后可记住实际尺寸，差异仍可能改变滚动位置。 */
```

配套文件：[long.html](scripts/21-rendering-and-performance/long.html)、[long.css](scripts/21-rendering-and-performance/long.css) · [浏览器预览](http://127.0.0.1:8101/scripts/21-rendering-and-performance/long.html)

## 7 transform、opacity 与 will-change 的限制

transform 和 opacity 常适合做不影响正常流尺寸的动画，但不能保证一定创建独立合成层，也不能保证每帧只合成。大图片的栅格化、纹理内存、混合、复杂裁剪以及设备资源都可能成为成本。

CSS 层叠上下文是绘制顺序规则，与浏览器为渲染选择的合成层不是一一对应。opacity 小于 1 会影响整体及其后代的透明度；即使为0也仍占位，并可能参与焦点与点击，不能用它替代真正的隐藏状态。

will-change 是可能优化的提示，不是强制加速命令。本例默认关闭，只在勾选后给第一项设置 transform 提示，用来比较工具中的层与成本；取消后回到 auto。先在 transform 模式下勾选并等待短暂准备时间，再运行相同动画，结束后取消。

长期给大量元素设置提示可能增加内存和管理成本，还会提前产生某些包含块或层叠上下文效果。只有发现实际问题、定位到对应元素后才尝试；如果测不到稳定收益，应去掉提示。不要为了“启用GPU”随处加入 translateZ(0)。

```html
<input id="fade" type="checkbox"><label for="fade">播放透明度变化</label>
<div class="fade-box">这段文字始终保留布局占位。</div>
```

```css
#hint:checked ~ .samples .row:first-child .mover { will-change: transform; }
@keyframes fade { from { opacity: 1; } to { opacity: 0.5; } }
.fade-box { width: 280px; padding: 12px; background: #cce8ed; animation: fade 2s linear infinite alternate paused; }
#fade:checked ~ .fade-box { animation-play-state: running; }
/* 取消hint后恢复will-change:auto；提示只用于试验，不预设它能改善此例。 */
```

配套文件：[index.html](scripts/21-rendering-and-performance/index.html)、[performance.css](scripts/21-rendering-and-performance/performance.css) · [浏览器预览](http://127.0.0.1:8101/scripts/21-rendering-and-performance/index.html)

## 8 用同一负载记录修改前后

先以首页位移对照完成一次可复查测量：

（1）固定浏览器版本、设备、电源状态、900×800 CSS像素视口和100%页面缩放；关闭无关页面。记录减少运动偏好，动画被关闭时不继续比较。保持透明度动画和 will-change 都关闭。

（2）刷新后选择 left 模式。打开 Performance，保持两次相同的 CPU 限速设置；先用无额外限速建立基线，再按需要另做一组4倍限速，不能混比。

（3）开始录制，勾选播放，经过至少两个完整往返后停止录制，再暂停。放大中间相同长度的稳定区间，查看 Main 中的 Recalculate Style、Layout、Paint 和相关合成工作；在 Summary 或 Bottom-up 中比较耗时分布，并查看 Frames 中是否有延迟或丢帧。

（4）刷新后选择 transform 模式，重复同样操作。每种模式做三次，交替顺序，保存原始 trace；记录区间长度、阶段耗时和帧情况。若差异很小或波动重叠，就如实记为“本负载下未观察到稳定差异”。

（5）另开 Rendering 中的 Paint flashing、Layer borders 或帧统计辅助定位，再关闭这些叠加层做正式比较；绿色重绘区域、层数或FPS显示都不是单独的总性能评分。工具界面随版本变化，按相同概念查找对应入口。

长页比较采用相同视口、同样缓存状态的重新加载，再沿相同路径滚动到第32条；既比较加载和滚动的阶段耗时，也检查查找、锚点与焦点是否正常。资源页则录制延迟显示前后的 Layout Shift，和位移动画分开结论。

本章成功标准是能从真实记录指出一次改动影响了哪个阶段，并同时保留可见行为；不是必须得到某个“更快百分比”。

```html
<input id="run" type="checkbox"><label for="run">播放</label>
```

```css
@media (prefers-reduced-motion: reduce) {
  .mover, .fade-box { animation: none; }
  #transform-mode:checked ~ .samples .mover { animation-name: none; }
  #hint:checked ~ .samples .row:first-child .mover { will-change: auto; }
}
/* 减少运动下不播放；测量动画前先确认当前偏好与播放状态。 */
```

配套文件：[index.html](scripts/21-rendering-and-performance/index.html)、[performance.css](scripts/21-rendering-and-performance/performance.css) · [浏览器预览](http://127.0.0.1:8101/scripts/21-rendering-and-performance/index.html)

## 本章小结

- 渲染由多个相关阶段组成，每帧所需工作取决于属性、内容和实现。
- 资源发现与字体度量可能影响首次显示和布局稳定性；预留空间要匹配实际内容。
- contain 改变边界条件，content-visibility 保留DOM并按需呈现内容。
- 合成友好的候选属性和 will-change 都需要测量，不能据属性名承诺零成本。
- 比较保持负载和环境一致，同时检查性能数据与交互行为。

## 练习

（1）按首页步骤完成 left 与 transform 的三组交替记录，写明一个实际差异或“未见稳定差异”，保留对应 trace；不要填入估计耗时。

（2）在资源页把预留宽高比临时改为 1 / 1，再录制延迟显示，观察留白或位置变化；解释为何“有预留”不等于“预留正确”。

（3）在 containment.html 中去掉 .explicit 的最小高度，观察边框盒高度；说明尺寸隔离为什么需要外部尺寸条件。

（4）对 long.html 使用页内查找定位第30节段落，再跳到第32条链接并用键盘聚焦；比较 baseline.html 的相同操作，同时记录滚动过程中实际发现。

### 提示

练习修改位于 scripts/21-rendering-and-performance/，结束后恢复。trace 中的阶段工作与用户看到的错误要分别说明，单独的CSS支持查询不能替代两者。

## 参考与引用来源

- web.dev：[Rendering performance](https://web.dev/articles/rendering-performance) 的样式、布局、绘制、栅格化和合成；[High-performance animations](https://web.dev/articles/animations-guide) 的属性选择与测量；[Extract critical CSS](https://web.dev/articles/extract-critical-css) 的阻塞样式与内联取舍；[Font best practices](https://web.dev/articles/font-best-practices) 的字体发现、资源链与布局；[Optimize CLS](https://web.dev/articles/optimize-cls) 的空间预留、输入宽限与偏移归因。
- Chrome for Developers：[Analyze runtime performance](https://developer.chrome.com/docs/devtools/performance) 的录制和CPU限速；[Performance reference](https://developer.chrome.com/docs/devtools/performance/reference) 的 Frames、Main、Bottom-up 与保存记录；[Rendering performance tools](https://developer.chrome.com/docs/devtools/rendering/performance) 的绘制闪烁和层边界；[Network](https://developer.chrome.com/docs/devtools/network) 的缓存、请求与Timing；[CSS reference](https://developer.chrome.com/docs/devtools/css/reference) 的Rendered Fonts。
- W3C：[CSS Containment 2 §3](https://www.w3.org/TR/css-contain-2/#containment-types) 的各种隔离及其副作用、[§4](https://www.w3.org/TR/css-contain-2/#content-visibility) 的跳过内容、可访问操作与实现空间。
- MDN：[contain](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/contain)、[content-visibility](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/content-visibility)、[contain-intrinsic-size](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/contain-intrinsic-size) 的取值、估计和支持范围；[will-change](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/will-change#description) 的提示、资源与使用限制；[opacity](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/opacity) 的整体透明度与交互边界；[font-display](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@font-face/font-display) 的描述符取值；[setTimeout](https://developer.mozilla.org/en-US/docs/Web/API/Window/setTimeout#reasons_for_longer_delays_than_specified) 的延迟不精确条件。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface) 的本地服务参数。